In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

d:\Programing\LangChain\langvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# creation  of langchain document  for ipl players
doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [3]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vector_store = Chroma(
    embedding_function=embedding_model,
    persist_directory="chroma_db",
    collection_name="sample"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5948.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
#add docs
vector_store.add_documents(docs)

['306ea529-36b0-4ecf-9938-9603221434bd',
 'e1ee8338-6a34-4937-ab61-d93c58eeb640',
 '1389c577-d588-45a8-b4a3-55a1289c55ba',
 'acf30fa5-5027-4c04-9428-d41f5f65a4ae',
 '1fd00973-06c8-414c-908d-90f33e111572']

In [6]:
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['306ea529-36b0-4ecf-9938-9603221434bd',
  'e1ee8338-6a34-4937-ab61-d93c58eeb640',
  '1389c577-d588-45a8-b4a3-55a1289c55ba',
  'acf30fa5-5027-4c04-9428-d41f5f65a4ae',
  '1fd00973-06c8-414c-908d-90f33e111572'],
 'embeddings': array([[ 0.00994723,  0.06914338, -0.05147114, ..., -0.0354334 ,
          0.01284808,  0.01248293],
        [ 0.00127743,  0.03129851, -0.02375379, ..., -0.00518362,
         -0.03280614,  0.02737717],
        [-0.10265917,  0.02650812,  0.02271503, ..., -0.03359742,
         -0.07984945, -0.01507708],
        [ 0.02123393, -0.02468549, -0.04494374, ..., -0.10995809,
          0.0057256 ,  0.09915379],
        [ 0.01873982,  0.04382847, -0.04304255, ..., -0.07801619,
         -0.07840681, -0.00304189]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [7]:
vector_store.similarity_search(
    query='Who among these are a captian?',
    k=2 #how many similar vector i want to see.
)

[Document(id='1389c577-d588-45a8-b4a3-55a1289c55ba', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
 Document(id='e1ee8338-6a34-4937-ab61-d93c58eeb640', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [8]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a captian?',
    k=2
)

[(Document(id='1389c577-d588-45a8-b4a3-55a1289c55ba', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.481388807296753),
 (Document(id='e1ee8338-6a34-4937-ab61-d93c58eeb640', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.5754146575927734)]

In [9]:
# meta data filtering
vector_store.similarity_search_with_score(
    query ='',
    filter = {"team": "Chennai Super Kings"}

)

[(Document(id='1389c577-d588-45a8-b4a3-55a1289c55ba', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436007499694824),
 (Document(id='1fd00973-06c8-414c-908d-90f33e111572', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.890937328338623)]